# Nonlinear propagation using a split-step approach

In this tutorial, we'll try to simulate the propagation of an intense laser pulse through a piece of fused silica glass, and observe the influence of the dispersion and self phase modulation that the pulse experiences in the material.

This should give an overview of the functionality of the `AngularSpectrumDFFTPropagator`, the nonlinear phase applied by the `NonlinearKerrStep` and how do combine them in a simple, iterative split-step method.

First, we'll import a few relevant functions and classes:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c

from lasy.laser import Laser
from lasy.profiles import GaussianProfile
from lasy.propagators import AngularSpectrumPropagator, NonlinearKerrStep
from lasy.utils.laser_utils import get_dispersion, get_spectrum

Next, we can define key parameters of the laser pulse and use them to create a Gaussian laser profile object.

In [ ]:
wavelength = 800e-9  # wavelength in meters
laser_energy = 0.5e-3  # laser pulse energy in Joule
tau = 30e-15  # pulse duration in seconds
w0 = 1e-3  # laser waist in meters
polarisation = (0, 1)  # polarisation vector of the laser
t_peak = 0  # laser peak position in time

profile = GaussianProfile(
    wavelength=wavelength,
    laser_energy=laser_energy,
    tau=tau,
    w0=w0,
    pol=polarisation,
    t_peak=t_peak,
)

To create the full laser object, we now define the parameters of the simulation grid, and combine the grid and profile to a laser object.

In [ ]:
dim = "xyt"
hi = (2.5e-3, 2.5e-3, 500e-15)
lo = (-2.5e-3, -2.5e-3, -500e-15)
npoints = (100, 100, 200)

laser = Laser(dim=dim, hi=hi, lo=lo, npoints=npoints, profile=profile)

To check that the pulse looks as expected, we can show its profile

In [ ]:
laser.show(envelope_type="intensity")

To see the influence of the self-phase modulation on the spectral properties of the laser, we can extract the initial spectrum and GDD curve of the laser pulse. 

In [ ]:
initial_spectrum, omega = get_spectrum(
    grid=laser.grid, dim=laser.dim, omega0=laser.profile.omega0
)

initial_gdd, initial_gdd0 = get_dispersion(
    grid=laser.grid, dim=laser.dim, omega0=laser.profile.omega0, order=2
)

Now let's define the properties of the fused silica material that the we want the pulse to propagate in. We will need the refractive index (ideally in form of the Sellmeier equation) and the intensity dependent refractive index, $n_2$. Both taken from [refractiveindex.info](https://www.refractiveindex.info).

At this point we can also define the thickness of the fused silica window that we want to propagate through. In this example we'll use a 5mm thick window.

In [ ]:
# refractive index
def n_fusedsilica(wavelength):
    """Sellmeier equation for fused silica."""
    x = wavelength * 1e6  #  convert wavelength to µm

    n = np.sqrt(
        1
        + 0.6961663 / (1 - (0.0684043 / x) ** 2)
        + 0.4079426 / (1 - (0.1162414 / x) ** 2)
        + 0.8974794 / (1 - (9.896161 / x) ** 2)
    )
    return n


# intensity dependent refractive index
n2 = 2.7e-20

# material thickness
d_FS = 5e-3

A very common and proven approach for accurately simulating the nonlinear propagation of laser pulses is the so-called split-step method. In this method, the linear propagation, that entails effects such as dispersion and diffraction, and the non-linear propagation (containing the intensity dependent effects) into two different propagation steps. 

For our purpose, we can use the `AngularSpectrumDFFTPropagator` that describes the linear propagation in dispersive media, and the `NonlinearKerrStep` for the nonlinear propagation step. We will create them in the following:

In [ ]:
linear_propagator = AngularSpectrumPropagator(
    omega0=profile.omega0, n=n_fusedsilica, dim=laser.dim
)
nonlinear_phase = NonlinearKerrStep(n2=n2, k0=laser.profile.omega0 / c)

To accurately simulate the interplay of the two propagation steps, the split-step approach devides total propagation distance into multiple sub-steps and alternately applies the two propagation steps. In general, the simulation result will become more accurate the smaller the propagation steps. In this specific case 10 steps with a size of $\frac{d_{FS}}{N_{steps}}=\frac{5mm}{10}=0.5mm$ seems like a good compromise between accurate results and a reasonably short simulation time.

Lets do this iterative propagation next:

In [ ]:
Nsteps = 10  # number of propagation steps
dz = d_FS / Nsteps  # length of the individual propagation steps

laser.add_propagator(linear_propagator)

for i in range(Nsteps):
    # linear propagation 1/2 step
    laser.propagate(dz / 2)

    # nonlinear propagation step
    laser.grid = nonlinear_phase.apply(grid_in=laser.grid, distance=dz)

    # linear propagation 1/2 step
    laser.propagate(dz / 2)
    # print progress
    print(f"Progress: {100 * (i + 1) / Nsteps:.1f}%", end="\r")

If we now again show the laser profile we can already see that the temporal shape of the pulse has changed:

In [ ]:
laser.show(envelope_type="intensity")

To see the effects on the spectral properties, we can again extract the spectrum and GDD curve to compare to the initial ones.

In [ ]:
output_spectrum, omega = get_spectrum(
    grid=laser.grid, dim=laser.dim, omega0=laser.profile.omega0
)

output_gdd, output_gdd0 = get_dispersion(
    grid=laser.grid, dim=laser.dim, omega0=laser.profile.omega0, order=2
)

Now lets plot them for comparison

In [ ]:
wavelength_nm = 1e9 * 2 * np.pi * c / omega

# spectrum
plt.plot(wavelength_nm, initial_spectrum, label="Input spectrum")
plt.plot(wavelength_nm, output_spectrum, label="Output spectrum")

plt.ylabel("Spectral intensity")
plt.xlabel("Wavelength [nm]")

plt.ylim(
    0,
)
plt.xlim(670, 950)

plt.legend(loc="upper left")
plt.twinx()

# GDD
plt.plot(wavelength_nm, initial_gdd * 1e30, label="Input GDD", linestyle="--")
plt.plot(wavelength_nm, output_gdd * 1e30, label="Output GDD", linestyle="--")

plt.ylabel("GDD [fs$^2$]")
plt.xlabel("Wavelength [nm]")

plt.ylim(-5000, 5000)

plt.legend(loc="upper right")
plt.show()

We can clearly see how the spectrum broadens and how the phase is affected by the intensity dependent refractive index and the resulting self-phase modulation.

Finally, we can let the laser propagate in vacuum for some more distance to show the influence of the self-focusing that the pulse experiences in the fused silica. The beam size decreases and the spatial profile is deformed from the ideal Gaussian that it is intially.

In [ ]:
# define propagator for vacuum
linear_propagator_vacuum = AngularSpectrumPropagator(
    omega0=profile.omega0, n=1.0, dim=laser.dim
)
laser.add_propagator(linear_propagator_vacuum)

# propagate in vacuum
laser.propagate(0.2)

# show laser after propagation
laser.show(envelope_type="intensity")

## Soliton propagation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import c

from lasy.laser import Laser
from lasy.profiles import CombinedLongitudinalTransverseProfile
from lasy.profiles.longitudinal import Sech2Profile
from lasy.profiles.transverse import PlaneWaveProfile
from lasy.utils.laser_utils import get_duration, get_laser_power

To verify the validity of the split-step approach, we can simulate the propagation of a soliton pulse, which is a special case where a $\mathrm{sech}^2$ shaped pulse keeps its temporal shape due to a balance of group velocity dispersion and self-phase modulation. This is typically found in fiber lasers, where the pulses are spatially confined and pulse durations are moderate, such that higher order dispersion plays a subordinate role.
In simple fused silica based fibers, this works only at relatively long wavelengths such as the 1.55µm commonly found in telecom fiber systems. This is because only at these wavelengths does fused silica introduce negative dispersion, that is required to balance the positive dispersion introduced by self-phase modulation.

We'll start by defining a laser pulse at this wavelength. To minimize the influence of higher order dispersion, we use a moderately long pulse duration of 1ps.

In [ ]:
# Physical Parameters
wavelength = 1550e-9  # wavelength in meters
tau = 1000e-15 / (
    2 * np.arccosh(np.sqrt(2))
)  # FWHM pulse duration (1ps) converted to the sech pulse duration
t_peak = 0.0  # peak position in time
pol = (1, 0)  # polarisation vector of the laser

To mimic the spatially confined propagation, we run our simulation in 1D, which can be done by combining `PlaneWaveProfile` in the transverse plane and a `Sech2Profile` in time.

In [ ]:
# longitudinal profile
long_prof = Sech2Profile(wavelength, tau, t_peak)

# transverse profile
tran_prof = PlaneWaveProfile()

# combined profile
profile = CombinedLongitudinalTransverseProfile(
    wavelength, pol, long_prof, tran_prof, peak_power=1.0
)

# Computational Grid
dim = "xyt"
lo = (None, None, -10 * tau)
hi = (None, None, 10 * tau)
npoints = (1, 1, 1000)

laser = Laser(dim, lo, hi, npoints, profile)

As a next step, we can define the parmeters of the fiber. In this case we take a 1km long fiber, with the correct intensity dependent refractive index $n_2$ at the 1.55µm wavelength.

In [ ]:
n2 = 2.41e-20  # intensity dependent refractive index
d_FS = 1000  # meters, total propagation distance in fused silica

Now that we have set the pulse and material properties, we can calculate the peak power of the pulse at which the soliton condition is met.
For this we need to calculate the dispersion parameter $\frac{\partial^2 k}{\partial \omega} = \beta_2$ of the fiber from its refractive index.

In [ ]:
from scipy.interpolate import interp1d


def get_beta2(n, omega, omega0):
    """Calculate the second order dispersion coefficient."""
    # ensure that omega is sorted
    order = np.argsort(omega)
    omega = omega[order]
    n = n[order]

    # calculate the wavevector k
    k = n * omega / c

    # second derivative of k with respect to omega
    beta2 = np.gradient(np.gradient(k, omega), omega)

    # evaluate at omega0
    beta2_0 = interp1d(omega, beta2, bounds_error=True)(omega0)
    return beta2_0

With this function we can calculate the dispersion parameter beta2 at the central frequency of the laser

In [ ]:
# create frequency axis
_, omega = laser.grid.get_spectral_field()
omega += laser.profile.omega0

# get refractive index
refractive_index = n_fusedsilica(2 * np.pi * c / omega)

# calculate the dispersion parameter
beta2 = get_beta2(refractive_index, omega, laser.profile.omega0)
print(f"Dispersion parameter: {beta2 * 1e27:.3f} fs^2/mm")

With this we can then calculate the soliton power at which SPM and dispersion are perfectly balanced, and normalize the laser power to this value.

In [ ]:
soliton_power = abs(beta2) * c / (tau**2 * n2 * laser.profile.omega0)
laser.normalize(soliton_power, kind="peak_power")

Check that the temporal shape is as expected.

In [ ]:
power = get_laser_power(laser.dim, laser.grid)

plt.figure()
plt.plot(laser.grid.axes[-1] * 1e15, power / 1e12)
plt.xlim(laser.grid.axes[-1][0] * 1e15, laser.grid.axes[-1][-1] * 1e15)
plt.ylim(0, None)
plt.xlabel("Time (fs)")
plt.ylabel("Instantaneous Power (TW)")

Define our propagator and nonlinear step:

In [ ]:
linear_propagator = AngularSpectrumPropagator(
    omega0=profile.omega0, n=n_fusedsilica, dim=laser.dim
)
nonlinear_phase = NonlinearKerrStep(n2=n2, k0=laser.profile.omega0 / c)

Run the actual split-step propagation:

In [ ]:
dz = 100e-3  # meters, step size

Nsteps = int(d_FS / dz)  # number of propagation steps

# add propagators to the laser
laser.add_propagator(linear_propagator)

peak_power = []
duration = []
shape = []
err_rel = []
temporal_field_initial = laser.grid.get_temporal_field()

# propagate using second order split step method
for i in range(Nsteps):
    laser.propagate(dz / 2)
    laser.grid = nonlinear_phase.apply(grid_in=laser.grid, distance=dz)
    laser.propagate(dz / 2)

    duration.append(get_duration(laser.grid, laser.dim, level=0.5))
    peak_power.append(np.max(get_laser_power(laser.dim, laser.grid)))
    shape.append(get_laser_power(laser.dim, laser.grid))

    err_abs = np.abs(
        abs(laser.grid.get_temporal_field()) - abs(temporal_field_initial)
    ).sum()
    err_norm = max(
        np.abs(laser.grid.get_temporal_field()).sum(),
        (np.abs(temporal_field_initial)).sum(),
    )

    if err_norm > 0:
        err_rel.append(err_abs / err_norm)
    else:
        err_rel.append(0.0)

    print(f"Progress: {100 * (i + 1) / Nsteps:.1f}%", end="\r")

duration = np.array(duration)
peak_power = np.array(peak_power)

shape = np.array(shape)

If we now plot the result, we can see that the pulse shape remains constant during the propagation.

In [ ]:
plt.imshow(
    shape.T[::10] * 1e-12,
    aspect="auto",
    extent=[0, d_FS, lo[-1] * 1e15, hi[-1] * 1e15],
    cmap="Reds",
)
plt.xlabel("Propagation distance (m)")
plt.ylabel("Time (fs)")
plt.colorbar(label="Peak intensity (TW/m$^2$)")

We can also look at the relative changes of the pulse duration and peak power during the kilometer of propagation, and see that they are constant to the sub-permille-level, as one would expect from the propagation of a perfect soliton pulse.

In [ ]:
z = np.linspace(0, d_FS, Nsteps)
fig, ax = plt.subplots(2, 1)

ax[0].plot(z, 1e15 * duration, label="Pulse duration", linestyle="-", color="C0")
ax[0].set_ylabel(r"Duration [fs]")
ax[0].legend(loc="upper left")
ax[0].set_ylim(duration.min() * 0.91 * 1e15, duration.max() * 1.11 * 1e15)
ax01 = ax[0].twinx()

ax01.plot(z, peak_power * 1e-12, label="Peak power", color="C1", linestyle="-")
ax01.set_ylabel("P [TW/m$^2$]")
ax01.legend(loc="upper right")
ax01.set_ylim(peak_power.min() * 0.9 * 1e-12, peak_power.max() * 1.1 * 1e-12)
ax01.set_xlim(z.min(), z.max())

ax[1].plot(z, duration / duration[0], label="Pulse duration")
ax[1].set_ylabel(r"$\tau/\tau_0$")
ax[1].legend(loc="upper left")
ax[1].set_ylim(1 - 2e-3, 1 + 2e-3)
ax[1].set_xlabel("Propagation distance (m)")
ax11 = ax[1].twinx()

ax11.plot(z, peak_power / peak_power[0], label="Peak power", color="C1")
ax11.set_ylabel("P/P$_0$")
ax11.legend(loc="upper right")
ax11.set_ylim(1 - 2e-3, 1 + 2e-3)
ax11.set_xlim(z.min(), z.max())